In [ ]:
from pathlib import Path
import subprocess
from google.colab import files

if not Path('/content/UNITAS').exists():
    subprocess.run(['git', 'clone', 'https://github.com/DexForce/UNITAS.git', '/content/UNITAS'], check=True)
print('Upload record_libero.py, replay_data.py and serve_timed.py from src/stage3_policy')
uploaded = files.upload()
folder = Path('/content/stage3_policy')
folder.mkdir(exist_ok=True)
assert {'record_libero.py', 'replay_data.py', 'serve_timed.py'} <= uploaded.keys()
for name, data in uploaded.items():
    (folder / Path(name).name).write_bytes(data)


In [ ]:
%%bash
set -e
if [ ! -x /content/miniforge/bin/conda ]; then
  curl -fL https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -o /content/Miniforge3.sh
  bash /content/Miniforge3.sh -b -p /content/miniforge
fi
/content/miniforge/bin/conda --version


In [ ]:
%%bash
set -e
if [ ! -x /content/miniforge/envs/unitas/bin/python ]; then
  /content/miniforge/bin/conda create -n unitas python=3.10 pip -y
fi
/content/miniforge/envs/unitas/bin/python -m pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
/content/miniforge/envs/unitas/bin/python -m pip install -e /content/UNITAS 'huggingface_hub[cli]>=0.34,<1'
if [ ! -d /content/LIBERO ]; then
  git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO
fi
if [ ! -x /content/miniforge/envs/libero/bin/python ]; then
  /content/miniforge/bin/conda create -n libero python=3.10 pip -y
fi
PY=/content/miniforge/envs/libero/bin/python
"$PY" -m pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu
"$PY" -m pip install numpy==1.26.4 robosuite==1.4.0 bddl==1.0.1 gym==0.25.2 easydict==1.9 cloudpickle==2.1.0 h5py pyyaml termcolor future==0.18.2 matplotlib
"$PY" -m pip install -r /content/UNITAS/evaluation/requirements.txt mujoco==3.2.3
"$PY" -m pip install --no-deps -e /content/LIBERO -e /content/UNITAS
export LIBERO_CONFIG_PATH=/content/libero-config
export PYTHONPATH=/content/LIBERO${PYTHONPATH:+:$PYTHONPATH}
mkdir -p /content/LIBERO/datasets
printf 'n\n' | "$PY" -c 'import libero.libero'


In [ ]:
%%bash
set -e
cd /content/UNITAS
/content/miniforge/bin/conda run --no-capture-output -n unitas hf download microsoft/Florence-2-large --local-dir checkpoints/Florence-2-large --exclude '*.bin'
/content/miniforge/bin/conda run --no-capture-output -n unitas hf download RobbinWang123/Unitas unitas_libero_finetuned.pt --local-dir checkpoints
/content/miniforge/envs/unitas/bin/python - <<'PY'
from pathlib import Path
p = Path('src/unitas/inference/policy.py')
text = p.read_text()
old = 'self.device.type, dtype=torch.bfloat16, enabled=self.device.type == "cuda"'
new = 'self.device.type, dtype=torch.bfloat16, enabled=False'
if old in text:
    assert text.count(old) == 1
    if not p.with_suffix('.py.original').exists():
        p.with_suffix('.py.original').write_text(text)
    p.write_text(text.replace(old, new))
else:
    assert new in text, 'Unexpected policy.py contents'
PY


In [ ]:
%%bash
set -eo pipefail
cd /content/UNITAS
export LIBERO_CONFIG_PATH=/content/libero-config
export PYTHONPATH=/content/LIBERO${PYTHONPATH:+:$PYTHONPATH}
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl MPLBACKEND=Agg
export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1
PY=/content/miniforge/envs/unitas/bin/python
SIM=/content/miniforge/envs/libero/bin/python
OUT="outputs/stage3/$(date -u +%Y%m%dT%H%M%S)"
mkdir -p "$OUT"
TASK=$("$SIM" -c 'from libero.libero import benchmark; s=benchmark.get_benchmark_dict()["libero_90"](); print(next(i for i in range(s.n_tasks) if s.get_task(i).language=="put the white mug on the plate"))' | tail -n 1)
export UNITAS_LIGHT_SCALE=1.0
export UNITAS_TIMING_PATH="$OUT/server-policy-timing.jsonl"
"$PY" -u /content/stage3_policy/serve_timed.py --domain libero --checkpoint checkpoints/unitas_libero_finetuned.pt --florence-dir checkpoints/Florence-2-large --device cuda --port 8000 > "$OUT/server.log" 2>&1 &
SERVER_PID=$!
trap 'kill "$SERVER_PID" 2>/dev/null || true; wait "$SERVER_PID" 2>/dev/null || true' EXIT
READY=0
for _ in $(seq 1 120); do
  kill -0 "$SERVER_PID" || { tail -n 30 "$OUT/server.log"; exit 1; }
  if curl -fsS http://127.0.0.1:8000/health > "$OUT/health.json"; then READY=1; break; fi
  sleep 5
done
test "$READY" -eq 1
"$SIM" -u /content/stage3_policy/record_libero.py --host 127.0.0.1 --port 8000 --suite libero_90 --task-id "$TASK" --num-episodes 1 --seed 42 --output-dir "$OUT" 2>&1 | tee "$OUT/client.log"
